# Measuring the sparsity gained with TEAL and WINA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2MOData/LLM-Activation-Sparsity-Analysis-with-Oracle-Activations-A-Research-and-Experimental-Project/blob/main/notebooks/sparsity_rates_teal_wina.ipynb)

TEAL and WINA are **threshold methods**: a neuron of the FFN is set to zero when its score is below
a threshold fixed for its layer. So we do not choose how many neurons are cut: the number depends on
the token and on the layer, and is only known after the forward pass. This notebook puts one hook on
the input of every `down_proj` of Qwen3-8B, calibrates the thresholds, and measures the **realized
sparsity**: per layer (local) and over the whole model (global).

**How to run it**

1. Open the notebook in Colab: click the badge above, or *File → Open notebook →
   GitHub*, paste the repo URL, pick your branch and select this notebook.
2. Select a GPU: *Runtime → Change runtime type → GPU*. Qwen3-8B in bf16 needs ~16 GB, more than a
   T4 (15 GB): use an L4 or an A100. On a T4 the notebook falls back to **Qwen3-4B** (same
   architecture, 36 layers), which is enough to check the code but not for the project's numbers.
3. If you are testing a branch, set `BRANCH` in the setup cell below.
4. *Runtime → Run all*. The first run takes ~5 min to download Qwen3-8B (16 GB), then calibration and
   measurement take a few minutes on an A100.

The results (thresholds, sparsity per layer) are written to `outputs/sparsity_rates/`. A Colab runtime
is wiped when it disconnects: set `SAVE_TO_DRIVE = True` in section 7 to keep them.

## 0. Setup

This cell does the following:
- clone the repo on Colab and check out the selected `BRANCH`.
- install the dependencies on Colab.

Rerun this cell every time you open a Colab session and after every push on GitHub.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/2MOData/LLM-Activation-Sparsity-Analysis-with-Oracle-Activations-A-Research-and-Experimental-Project.git"
BRANCH = "main"  # put your branch here to test code that is not merged yet

if IN_COLAB:
    # Clone the repo (or update it), then install the procom package and its dependencies
    !git clone -q {REPO_URL} /content/procom 2>/dev/null
    !cd /content/procom && git fetch -q && git checkout -q {BRANCH} && git pull -q
    !pip install -q -e /content/procom
    sys.path.insert(0, "/content/procom/src")  # makes procom importable without restarting

In [ ]:
import torch

import procom  # noqa: F401  (checks that the repo's package is installed)

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → GPU"
print(torch.cuda.get_device_name(0))

## 1. Loading Qwen3-8B

Only the target model is needed here, not the oracle. It is loaded in bf16 and **never in 4 or 8
bits**: quantization changes the activations themselves, which would bias the measured sparsity.

In each layer, the FFN computes `down_proj(a)` with `a = SiLU(gate_proj(x)) * up_proj(x)`, a vector of
12,288 values: one per **neuron**. Setting `a_j` to zero means that column `j` of `W_down` is not
needed for this token: this is the sparsity TEAL and WINA exploit, and the one we measure.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
# Qwen3-8B does not fit on a T4 in bf16: Qwen3-4B has the same architecture, to check the code
MODEL_NAME = "Qwen/Qwen3-8B" if gpu_memory_gb >= 20 else "Qwen/Qwen3-4B"
# T4s have no bf16 units: fall back to float16 there
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE, device_map="auto")
model.eval()
torch.set_grad_enabled(False)
device = model.device
layers = model.model.layers  # the transformer blocks; each has an FFN in `.mlp`
N_LAYERS = len(layers)
N_NEURONS = model.config.intermediate_size

print(
    f"{MODEL_NAME} | {DTYPE} | GPU {gpu_memory_gb:.0f} GB | {N_LAYERS} layers x {N_NEURONS} neurons"
)

## 2. The two scores

Both methods give each neuron `j` a score and cut the neurons whose score is below the threshold:

| Method | Score of neuron j | What it measures |
|---|---|---|
| **TEAL** ([arXiv:2408.14690](https://arxiv.org/abs/2408.14690)) | `|a_j|` | size of the activation |
| **WINA** ([arXiv:2505.19427](https://arxiv.org/abs/2505.19427)) | `|a_j| · ‖w_j‖` | contribution of the neuron to the layer's output |

`w_j` is column `j` of `W_down`. WINA is used **without its rotation**: the rotation changes the basis,
and the neurons would no longer be those of the original model.

In [ ]:
METHODS = ("teal", "wina")

# ‖w_j‖ for every layer: how much neuron j can move the output of its layer (used by WINA)
COLUMN_NORMS = [layer.mlp.down_proj.weight.float().norm(dim=0) for layer in layers]


def neuron_scores(acts, method, layer):
    """Score of every neuron (float32, same shape as `acts`): the lowest ones are cut."""
    scores = acts.abs().float()
    if method == "wina":
        scores = scores * COLUMN_NORMS[layer].to(scores.device)
    return scores

## 3. The hook

A *forward pre-hook* on `down_proj` receives its input `a` before the layer runs and can replace it.
The same hook serves both methods: it computes the scores, keeps the neurons whose score reaches the
layer's threshold, and records, **for every token**, the fraction of neurons it cut.

It also records the relative error it causes on the FFN's output, `‖W_down (a − a ⊙ m)‖ / ‖W_down a‖`
(`m` = the mask of kept neurons), summed over the tokens. This is the quantity the thresholds are
calibrated on in the next section.

Prompts are passed one at a time (batch of 1), so no token is padding and every token counts.

In [ ]:
import contextlib
from collections import defaultdict
from dataclasses import dataclass, field

import torch.nn.functional as F


@dataclass
class SparsityRecord:
    """What the hook saw, layer by layer, over all the forward passes of a `sparsify` block."""

    cut: dict = field(
        default_factory=lambda: defaultdict(list)
    )  # layer -> [cut fraction per token]
    error_sq: dict = field(default_factory=lambda: defaultdict(float))  # layer -> Σ ‖W (a − a⊙m)‖²
    output_sq: dict = field(default_factory=lambda: defaultdict(float))  # layer -> Σ ‖W a‖²

    def per_token(self, layer):
        """Fraction of the layer's neurons cut, for each token seen."""
        return torch.cat(self.cut[layer])

    def relative_error(self, layer):
        return (self.error_sq[layer] / self.output_sq[layer]) ** 0.5


@contextlib.contextmanager
def sparsify(method, thresholds, record):
    """Inside this block, every neuron whose score is below its layer's threshold is set to zero.

    thresholds: one threshold per layer (0 cuts nothing). record: a SparsityRecord to fill.
    """

    def make_hook(layer):
        def hook(module, args):
            acts = args[0]
            keep = neuron_scores(acts, method, layer) >= thresholds[layer]
            record.cut[layer].append((~keep).float().mean(dim=-1).flatten().cpu())
            removed = F.linear(acts * ~keep, module.weight)  # what the cut neurons would have added
            record.error_sq[layer] += removed.float().pow(2).sum().item()
            record.output_sq[layer] += F.linear(acts, module.weight).float().pow(2).sum().item()
            return (acts * keep,)  # replaces the input of down_proj

        return hook

    handles = [
        layer.mlp.down_proj.register_forward_pre_hook(make_hook(i))
        for i, layer in enumerate(layers)
    ]
    try:
        yield record
    finally:
        for handle in handles:
            handle.remove()  # always remove the hooks, otherwise they stay active afterwards

## 4. Calibrating the thresholds

There is **no fixed budget per layer**. For each layer, we take the largest threshold that keeps the
relative error of the FFN's output under a tolerance `b`, measured on a few thousand tokens of generic
text (WikiText-2, train split). This rule comes from SATS ([arXiv:2607.08991](https://arxiv.org/abs/2607.08991)).

Each layer is calibrated on the activations of the **dense** model, independently of the others. In a
sparse forward pass, the layers after the first cut receive slightly different inputs: the realized
sparsity and error are therefore not exactly those of the calibration, which is why section 5 measures
them on other text.

The search is a bisection on the number `k` of entries cut (the `k` lowest scores among all the
calibration tokens): cutting more entries is assumed to never reduce the error.

In [ ]:
from datasets import load_dataset

TOLERANCES = (0.02, 0.05, 0.10, 0.20, 0.30)  # b: allowed relative error of the FFN's output
SEQ_LEN = 512
N_CALIB_SEQS = 8  # 8 x 512 = 4,096 calibration tokens


def wikitext_chunks(split, n_chunks, seq_len=SEQ_LEN, n_chars=400_000):
    """`n_chunks` runs of `seq_len` consecutive WikiText-2 tokens, shape (n_chunks, seq_len)."""
    text = "".join(load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split=split)["text"])
    ids = tokenizer(text[:n_chars], return_tensors="pt").input_ids[0]
    assert len(ids) >= n_chunks * seq_len, "not enough text: raise n_chars"
    return ids[: n_chunks * seq_len].view(n_chunks, seq_len)


calib_ids = wikitext_chunks("train", N_CALIB_SEQS)
print("calibration tokens:", calib_ids.numel())

In [ ]:
from tqdm.auto import tqdm


def calibrate_layer(acts, weight, method, layer, tolerances, resolution=1e-4):
    """For each tolerance b, the largest threshold whose relative error stays <= b.

    acts: inputs of down_proj on the calibration tokens, (n_tokens, n_neurons).
    """
    scores = neuron_scores(acts, method, layer)
    sorted_scores = scores.flatten().sort().values
    output_norm = F.linear(acts, weight).float().norm()

    def relative_error(k):  # error when the k lowest scores are cut
        cut = scores < sorted_scores[k]
        return (F.linear(acts * cut, weight).float().norm() / output_norm).item()

    thresholds = {}
    for b in tolerances:
        low, high = 0, len(sorted_scores) - 1  # relative_error(0) = 0 <= b
        if relative_error(high) <= b:
            low = high
        while (high - low) / len(sorted_scores) > resolution:
            middle = (low + high) // 2
            if relative_error(middle) <= b:
                low = middle
            else:
                high = middle
        thresholds[b] = sorted_scores[low].item()
    return thresholds


def calibrate(calib_ids, tolerances=TOLERANCES):
    """thresholds[method][b] = list of one threshold per layer, from one dense forward pass."""
    thresholds = {m: {b: [None] * N_LAYERS for b in tolerances} for m in METHODS}
    progress = tqdm(total=N_LAYERS, desc="calibrating")

    def make_hook(layer):
        def hook(module, args):
            acts = args[0].flatten(0, -2)  # (n_tokens, n_neurons); the input is left unchanged
            for method in METHODS:
                found = calibrate_layer(acts, module.weight, method, layer, tolerances)
                for b, threshold in found.items():
                    thresholds[method][b][layer] = threshold
            progress.update()

        return hook

    handles = [
        layer.mlp.down_proj.register_forward_pre_hook(make_hook(i))
        for i, layer in enumerate(layers)
    ]
    try:
        model(calib_ids.to(device))  # every sequence has SEQ_LEN real tokens: no padding
    finally:
        for handle in handles:
            handle.remove()
        progress.close()
    return thresholds


thresholds = calibrate(calib_ids)
for method in METHODS:
    for b in TOLERANCES:
        t = thresholds[method][b]
        print(f"{method} b={b:.2f}: thresholds from {min(t):.3g} to {max(t):.3g}")

## 5. Measuring the realized sparsity

The thresholds are now fixed. We run the model with the hook on text it was **not** calibrated on:
WikiText-2 test sequences, and a few prompts per theme of the project's dataset (facts, context,
calculation, predictable text, sentiment, role), passed through the chat template as the oracle
experiments will do. These prompts are placeholders until `procom.dataset` exists.

- **Local sparsity** of a layer: fraction of its neurons cut, averaged over the tokens.
- **Global sparsity**: fraction of all the FFN neurons of the model cut. Every layer has the same number
  of neurons, so it is the mean of the local sparsities.

In [ ]:
EVAL_PROMPTS = {
    "facts": [
        "The Eiffel Tower is located in the city of",
        "The chemical symbol for gold is",
    ],
    "context": [
        "When Mary and John went to the store, John gave a drink to",
        "After Alice and Bob finished the project, Bob sent the report to",
    ],
    "calculation": [
        "23 + 48 =",
        "A shop sells pens at 3 euros each. Tom buys 7 pens. How much does he pay?",
    ],
    "predictable": [
        "Monday, Tuesday, Wednesday, Thursday,",
        "for i in range(10):\n    print(",
    ],
    "sentiment": [
        "This movie was a complete waste of time: the plot made no sense and the acting was awful.",
        "A warm, funny and beautifully shot film. I loved every minute of it.",
    ],
    "role": [
        "You are a pirate. Describe your ship in two sentences.",
        "You are a strict math teacher. Explain what a prime number is.",
    ],
}


def chat_ids(prompt):
    """Token ids of a one-message conversation, shape (1, n_tokens)."""
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, enable_thinking=False
    )
    return tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids


EVAL_SET = {"wikitext": list(wikitext_chunks("test", 4).split(1))}  # 4 x 512 tokens
EVAL_SET |= {theme: [chat_ids(p) for p in prompts] for theme, prompts in EVAL_PROMPTS.items()}
print({theme: sum(ids.numel() for ids in prompts) for theme, prompts in EVAL_SET.items()})

In [ ]:
records = {}  # (method, b, theme) -> SparsityRecord
for method in METHODS:
    for b in tqdm(TOLERANCES, desc=method):
        for theme, prompts in EVAL_SET.items():
            with sparsify(method, thresholds[method][b], SparsityRecord()) as record:
                for ids in prompts:
                    model(ids.to(device))  # one prompt at a time: no padding
            records[method, b, theme] = record

# Sanity check: with all thresholds at 0, nothing is cut
with sparsify("teal", [0.0] * N_LAYERS, SparsityRecord()) as dense:
    model(EVAL_SET["facts"][0].to(device))
assert all(dense.per_token(layer).max() == 0 for layer in range(N_LAYERS))

In [ ]:
import pandas as pd

rows = []
for (method, b, theme), record in records.items():
    for layer in range(N_LAYERS):
        per_token = record.per_token(layer)
        rows.append(
            {
                "method": method,
                "b": b,
                "theme": theme,
                "layer": layer,
                "sparsity": per_token.mean().item(),
                "sparsity_p10": per_token.quantile(0.1).item(),  # spread over the tokens
                "sparsity_p90": per_token.quantile(0.9).item(),
                "relative_error": record.relative_error(layer),
                "n_tokens": len(per_token),
            }
        )
per_layer = pd.DataFrame(rows)

# Global sparsity: mean over the layers (same number of neurons per layer), per (method, b, theme)
per_theme = (
    per_layer.groupby(["method", "b", "theme"])
    .agg(global_sparsity=("sparsity", "mean"), mean_error=("relative_error", "mean"))
    .reset_index()
)
# Over all the evaluation tokens, each theme weighted by its number of tokens
per_layer["cut_tokens"] = per_layer["sparsity"] * per_layer["n_tokens"]
overall = per_layer.groupby(["method", "b"])[["cut_tokens", "n_tokens"]].sum()
overall = (overall["cut_tokens"] / overall["n_tokens"]).rename("global_sparsity").reset_index()
per_layer = per_layer.drop(columns="cut_tokens")

print("Global sparsity, all evaluation tokens:")
print(overall.pivot(index="b", columns="method", values="global_sparsity").round(3))
print("\nGlobal sparsity per theme:")
print(
    per_theme.pivot_table(index=["method", "b"], columns="theme", values="global_sparsity").round(3)
)

Only the input of `down_proj` is made sparse here, so the global sparsity is the fraction of `W_down`
columns that a token does not need, i.e. about a third of the FFN's weights at most (`gate_proj` and
`up_proj` are still fully computed).

## 6. Figures

1. Local sparsity per layer, one curve per tolerance `b` (all evaluation tokens).
2. Relative error per layer in the sparse pass, compared with the tolerance it was calibrated for.
3. Spread over the tokens: global sparsity of each token, for `b = 0.10`.

In [ ]:
import matplotlib.pyplot as plt

all_tokens = (
    per_layer.assign(cut=per_layer["sparsity"] * per_layer["n_tokens"])
    .groupby(["method", "b", "layer"])[["cut", "n_tokens"]]
    .sum()
)
all_tokens = (all_tokens["cut"] / all_tokens["n_tokens"]).rename("sparsity").reset_index()
errors = per_layer[per_layer["theme"] == "wikitext"]

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
for column, method in enumerate(METHODS):
    for b in TOLERANCES:
        curve = all_tokens[(all_tokens["method"] == method) & (all_tokens["b"] == b)]
        axes[0, column].plot(curve["layer"], curve["sparsity"], marker=".", label=f"b = {b:.2f}")
        error = errors[(errors["method"] == method) & (errors["b"] == b)]
        line = axes[1, column].plot(error["layer"], error["relative_error"], marker=".")
        axes[1, column].axhline(b, color=line[0].get_color(), linestyle=":", linewidth=1)
    axes[0, column].set(title=f"{method.upper()}: local sparsity", ylabel="fraction of neurons cut")
    axes[1, column].set(
        title=f"{method.upper()}: FFN output error (WikiText test; dotted: b)",
        xlabel="layer",
        ylabel="relative error",
    )
    axes[0, column].set_ylim(0, 1)
axes[0, 0].legend()
fig.tight_layout()
plt.show()

B_SPREAD = 0.10
fig, ax = plt.subplots(figsize=(7, 3.5))
for method in METHODS:
    record_tokens = [
        torch.stack(
            [records[method, B_SPREAD, theme].per_token(layer) for layer in range(N_LAYERS)]
        )
        for theme in EVAL_SET
    ]
    per_token_global = torch.cat(record_tokens, dim=1).mean(dim=0)  # one value per token
    ax.hist(per_token_global.numpy(), bins=40, alpha=0.6, label=method.upper())
ax.set(
    title=f"Global sparsity of each token (b = {B_SPREAD:.2f})",
    xlabel="fraction of the model's FFN neurons cut",
    ylabel="tokens",
)
ax.legend()
fig.tight_layout()
plt.show()

## 7. Saving the results

- `thresholds.json`: the calibrated thresholds, to reuse them without recalibrating;
- `sparsity_per_layer.csv`: one row per (method, b, theme, layer);
- `sparsity_global.csv`: global sparsity per (method, b), over all the evaluation tokens and per theme.

In [ ]:
import json
import shutil
from pathlib import Path

SAVE_TO_DRIVE = False  # True: copy the results to Google Drive (Colab only)
OUTPUT_DIR = Path("outputs/sparsity_rates")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

settings = {
    "model": MODEL_NAME,
    "dtype": str(DTYPE),
    "hook": "forward pre-hook on mlp.down_proj (input = SiLU(gate) * up)",
    "calibration": f"WikiText-2 train, {calib_ids.numel()} tokens, each layer on dense activations",
    "tolerances": list(TOLERANCES),
}
thresholds_json = {m: {str(b): t for b, t in by_b.items()} for m, by_b in thresholds.items()}
(OUTPUT_DIR / "thresholds.json").write_text(
    json.dumps({"settings": settings, "thresholds": thresholds_json}, indent=1)
)
per_layer.to_csv(OUTPUT_DIR / "sparsity_per_layer.csv", index=False)
pd.concat([overall.assign(theme="all"), per_theme]).to_csv(
    OUTPUT_DIR / "sparsity_global.csv", index=False
)
print("saved to", OUTPUT_DIR.resolve())

if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    shutil.copytree(OUTPUT_DIR, "/content/drive/MyDrive/procom/sparsity_rates", dirs_exist_ok=True)